In [ ]:
import requests
import pandas as pd


In [ ]:
base_url = "https://api.enerpricedata.com"
api_key = ""           # Replace with your actual API key


In [ ]:
assert api_key, "Set api_key on the previous cell before continuing."

headers = {
    "X-API-Key": api_key  # FastAPI expects the key in this custom header
}


### Energy Futures:

EXCEL

In [ ]:
endpoint = "/datasets/download/energy-futures"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "end_operating_date": "2025-08-28",          # Optional, for date range
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "block_types": "7x8,2x16",                   # Optional: e.g., 7x8,2x16,5x16, etc. e.g., "7x8","2x16","5x16" -- By Default ALL 
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
end_op_date = params.get("end_operating_date", params["start_operating_date"])
filename = f"EPD_EnergyFutures_{control_area}_{end_op_date}.xlsx"


if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully and saved as '{filename}'")
else:
    # If an error occurs, print status and response text
    print(f"Error {response.status_code}: {response.text}")

CSV:

In [ ]:
endpoint = "/datasets/download/energy-futures/csv"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "block_types": "",                           # Optional: e.g., 7x8,2x16,5x16, etc. e.g., "7x8","2x16","5x16" -- By Default ALL 
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_EnergyFutures_{control_area}_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully and saved as '{filename}'")
else:
    # status and response text
    print(f"Error {response.status_code}: {response.text}")

JSON:

### Understanding Pagination Parameters

### Skip and Limit Explained

The JSON endpoints support pagination using `skip` and `limit` parameters:

- **`skip`**: Number of records to skip before starting to return results (offset)
- **`limit`**: Maximum number of records to return (default: 100, max: 1000)

### Examples:
- `skip=0, limit=100` → Returns records 1-100 (first page)
- `skip=100, limit=100` → Returns records 101-200 (second page)
- `skip=200, limit=50` → Returns records 201-250
- `skip=500, limit=100` → Returns records 501-600

#### Formula:
**Page Number = (skip ÷ limit) + 1**

In [ ]:
## Pagination Example: Getting All Records
#
# `skip` and `limit` only take effect when `raw=true`. The response shape is
# {"data": [...], "total": N, "page": p, "size": s}, so we paginate on
# payload["data"], not the top-level object.

import requests

endpoint = "/datasets/download/energy-futures/json"
all_records = []
skip = 0
limit = 100  # max 1000

while True:
    params = {
        "start_operating_date": "2025-08-25",
        "control_area": "ERCOT",
        "raw": True,       # required for skip/limit to apply
        "skip": skip,
        "limit": limit,
    }

    response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        break

    payload = response.json()
    page = payload.get("data", [])

    if not page:
        print(f"Pagination complete. Total records collected: {len(all_records)}")
        break

    all_records.extend(page)
    print(
        f"Page {payload.get('page', skip // limit + 1)}: "
        f"retrieved {len(page)} records "
        f"(total: {len(all_records)} of {payload.get('total', '?')})"
    )

    if len(page) < limit:
        print(f"Reached end of data. Total records: {len(all_records)}")
        break

    skip += limit

print(f"\nFinal Result: Collected {len(all_records)} total records")


In [ ]:
## Energy Futures JSON Download (Enhanced with Raw Output)

import json

endpoint = "/datasets/download/energy-futures/json"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "block_types": "7x8,2x16",                   # Optional: e.g., 7x8,2x16,5x16, etc. e.g., "7x8","2x16","5x16" -- By Default ALL 
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01",                    # Optional: filter end
    "raw": False,                                # Optional: if True, return raw JSON instead of file
    "skip": 0,                                   # Optional: pagination skip
    "limit": 100                                 # Optional: pagination limit (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_EnergyFutures_{control_area}_{start_op_date}.json"

if response.status_code == 200:
    if params.get("raw"):
        # If raw=True, response is JSON data directly
        data = response.json()
        print(f" Raw JSON data received: {len(data)} records")
        print(f"Sample record: {data[0] if data else 'No data'}")
    else:
        # If raw=False or not specified, response is a file
        data = response.json()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


### REC/RPS Downloads

The REC/RPS dataset provides Renewable Energy Certificate and Renewable Portfolio Standard data.


#### Excel

In [ ]:
## REC/RPS Excel Download

endpoint = "/datasets/download/rec-rps"

params = {
    "start_operating_date": "2025-08-01",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
end_op_date = params.get("end_operating_date", params["start_operating_date"])
filename = f"EPD_REC_RPS_{control_area}_{end_op_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


#### CSV

In [ ]:
## REC/RPS CSV Download

endpoint = "/datasets/download/rec-rps/csv"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_REC_RPS_{control_area}_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


#### JSON

In [ ]:
## REC/RPS JSON Download

import json

endpoint = "/datasets/download/rec-rps/json"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01",                    # Optional: filter end
    "raw": False,                                # Optional: if True, return raw JSON instead of file
    "skip": 0,                                   # Optional: pagination skip
    "limit": 100                                 # Optional: pagination limit (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_REC_RPS_{control_area}_{start_op_date}.json"

if response.status_code == 200:
    if params.get("raw"):
        # If raw=True, response is JSON data directly
        data = response.json()
        print(f" Raw JSON data received: {len(data)} records")
        print(f"Sample record: {data[0] if data else 'No data'}")
    else:
        # If raw=False or not specified, response is a file
        data = response.json()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


### Ancillary Uplift Downloads

The Ancillary Uplift dataset provides data on ancillary services and uplift charges in electricity markets.


#### Excel:

In [ ]:
## Ancillary Uplift Excel Download

endpoint = "/datasets/download/ancillary-uplift"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "end_operating_date": "2025-08-25",          # Optional, for date range
    "control_area": "PJM",                       # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
end_op_date = params.get("end_operating_date", params["start_operating_date"])
filename = f"EPD_AncillaryUplift_{control_area}_{end_op_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


#### CSV

In [ ]:
## Ancillary Uplift CSV Download

endpoint = "/datasets/download/ancillary-uplift/csv"

params = {
    "start_operating_date": "2025-08-25",        # Required
    "control_area": "PJM",                       # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01"                     # Optional: filter end
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_AncillaryUplift_{control_area}_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


#### JSON

In [ ]:
## Ancillary Uplift JSON Download

import json

endpoint = "/datasets/download/ancillary-uplift/json"

params = {
    "start_operating_date": "2025-08-29",        # Required
    "control_area": "ERCOT",                     # ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: filter start
    "end_date": "2030-09-01",                    # Optional: filter end
    "raw": False,                                # Optional: if True, return raw JSON instead of file
    "skip": 0,                                   # Optional: pagination skip
    "limit": 100                                 # Optional: pagination limit (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

control_area = params["control_area"]
start_op_date = params["start_operating_date"]
filename = f"EPD_AncillaryUplift_{control_area}_{start_op_date}.json"

if response.status_code == 200:
    if params.get("raw"):
        # If raw=True, response is JSON data directly
        data = response.json()
        print(f" Raw JSON data received: {len(data)} records")
        print(f"Sample record: {data[0] if data else 'No data'}")
    else:
        # If raw=False or not specified, response is a file
        data = response.json()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


### Utility Price:

Published on the 20th of each month, or the next business day when the 20th falls on a weekend or holiday.

All examples below use the publication date `2026-08-20`.


#### Excel

In [ ]:
endpoint = "/datasets/download/utility-price"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
    "end_operating_date": "2026-08-20",          # Optional, for date range
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

end_op_date = params.get("end_operating_date", params["start_operating_date"])
filename = f"EPD_UtilityPrice_{end_op_date}.xlsx"


if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully and saved as '{filename}'")
else:
    # If an error occurs, print status and response text
    print(f"Error {response.status_code}: {response.text}")

#### CSV — Summary only

Summary rows only, as a single CSV file. Accepts a single Publication Date; date ranges are Excel-only.


In [ ]:
## Utility Price — Summary CSV Download
# Endpoint tested live on publication date 2026-08-20.
# Published on the 20th of each month, or the next business day when the 20th falls on a weekend or holiday.

endpoint = "/datasets/download/utility-price/summary/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format; single date only
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_UtilityPrice_Summary_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Summary CSV downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")


#### CSV — Details only

Details rows only, as a single CSV file. Accepts a single Publication Date; date ranges are Excel-only.


In [ ]:
## Utility Price — Details CSV Download
# Endpoint tested live on publication date 2026-08-20.
# Published on the 20th of each month, or the next business day when the 20th falls on a weekend or holiday.

endpoint = "/datasets/download/utility-price/details/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format; single date only
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_UtilityPrice_Details_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Details CSV downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")


#### CSV — Combined (summary + details, ZIP)


In [ ]:
endpoint = "/datasets/download/utility-price/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
    #"end_operating_date": "2026-08-20",          # Optional, for date range
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

end_op_date = params.get("end_operating_date", params["start_operating_date"])
filename = f"EPD_UtilityPrice_{end_op_date}.zip"


if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully and saved as '{filename}'")
else:
    #status and response text
    print(f"Error {response.status_code}: {response.text}")

#### JSON


In [ ]:
import json

endpoint = "/datasets/download/utility-price/json"

params = {
    "start_operating_date": "2026-08-20",        # Required
    "raw": False,                                # Optional: if True, return raw JSON instead of file
    "skip": 0,                                   # Optional: pagination skip
    "limit": 100                                 # Optional: pagination limit (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_UtilityPrice_{start_op_date}.json"

if response.status_code == 200:
    if params.get("raw"):
        # If raw=True, response is JSON data directly
        data = response.json()
        print(f" Raw JSON data received: {len(data)} records")
        print(f"Sample record: {data[0] if data else 'No data'}")
    else:
        # If raw=False or not specified, response is a file
        data = response.json()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f" File downloaded successfully and saved as '{filename}'")
else:
    print(f" Error {response.status_code}: {response.text}")


### Natural Gas Utility Price

The Natural Gas Utility Price dataset returns both summary and detailed pricing rows for natural-gas utilities. The endpoints mirror the electric Utility Price endpoints but use the `/naturalgas-utility-price` prefix.

#### Excel

Returns a single workbook containing both summary and details sheets. Pass `end_operating_date` to get a ZIP of per-date xlsx files for a range.

In [ ]:
endpoint = "/datasets/download/naturalgas-utility-price"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
    "end_operating_date": "2026-08-20",          # Optional — when set, response is a ZIP of per-date xlsx files
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

end_op_date = params.get("end_operating_date", params["start_operating_date"])
is_range = params.get("end_operating_date") and params["end_operating_date"] != params["start_operating_date"]
filename = f"EPD_NaturalGasUtilityPrice_{end_op_date}." + ("zip" if is_range else "xlsx")

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")

#### CSV — Summary only

In [ ]:
endpoint = "/datasets/download/naturalgas-utility-price/summary/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_NaturalGasUtilityPrice_Summary_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Summary CSV downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")

#### CSV — Details only

In [ ]:
endpoint = "/datasets/download/naturalgas-utility-price/details/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_NaturalGasUtilityPrice_Details_{start_op_date}.csv"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Details CSV downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")

#### CSV — Combined (summary + details, ZIP)

In [ ]:
endpoint = "/datasets/download/naturalgas-utility-price/csv"

params = {
    "start_operating_date": "2026-08-20",        # Required in YYYY-MM-DD format
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_NaturalGasUtilityPrice_{start_op_date}.zip"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Combined CSV ZIP downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")

#### JSON

Default response is a downloadable `.json` file. Set `raw=true` for an inline JSON response with paginated `details` (use `offset`/`limit` to walk the pages — note: this endpoint uses `offset`, not `skip`).

In [ ]:
import json

endpoint = "/datasets/download/naturalgas-utility-price/json"

params = {
    "start_operating_date": "2026-08-20",        # Required
    "raw": False,                                # Optional: if True, return inline JSON
    "offset": 0,                                 # Optional: details-page offset (only when raw=True)
    "limit": 100                                 # Optional: details-page size (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_op_date = params["start_operating_date"]
filename = f"EPD_NaturalGasUtilityPrice_{start_op_date}.json"

if response.status_code == 200:
    if params.get("raw"):
        data = response.json()
        print(f"Summary rows: {len(data.get('summary', []))}")
        pag = data.get("pagination", {})
        print(f"Details page: {len(data.get('details', []))} of {pag.get('total')} (offset={pag.get('offset')}, limit={pag.get('limit')})")
    else:
        data = response.json()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f"File downloaded successfully and saved as '{filename}'")
else:
    print(f"Error {response.status_code}: {response.text}")

### Natural Gas Futures
Monthly natural gas forward curve for a publication date, across all settlement points. Download in Excel, CSV, or JSON.

#### Excel

In [ ]:
endpoint = "/datasets/download/naturalgas-futures"

params = {
    "start_operating_date": "2025-08-15",   # Required: curve publication date
    "settlement_point": "",                  # Optional: single settlement point; blank = all
    "start_date": "",                        # Optional: curve delivery start filter
    "end_date": "",                          # Optional: curve delivery end filter
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

filename = f"EPD_NaturalGasFutures_{params['start_operating_date']}.xlsx"
if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Saved {filename}")
else:
    print(f"Error {response.status_code}: {response.text}")


#### CSV

In [ ]:
endpoint = "/datasets/download/naturalgas-futures/csv"

params = {"start_operating_date": "2025-08-15"}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

filename = f"EPD_NaturalGasFutures_{params['start_operating_date']}.csv"
if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Saved {filename}")
else:
    print(f"Error {response.status_code}: {response.text}")


#### JSON

In [ ]:
import json

endpoint = "/datasets/download/naturalgas-futures/json"

params = {
    "start_operating_date": "2025-08-15",   # Required
    "raw": False,                            # True returns inline paginated JSON
    "skip": 0,                               # only used when raw=True
    "limit": 100,                            # only used when raw=True (max 1000)
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    if params["raw"]:
        print(f"Page {data['page']}: {data['size']} of {data['total']} records")
    else:
        with open("EPD_NaturalGasFutures.json", "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print("Saved EPD_NaturalGasFutures.json")
else:
    print(f"Error {response.status_code}: {response.text}")


### Fair Market Pricing (FMP)
All-in fair price for a single account, by cost component and by month. Live for **PJM** and **MISO**. Limits: 10 requests/minute, 600/day. `unit` sets the basis (`MWh` default, or `kWh`).

#### Pricing options (cascading dropdowns)

In [ ]:
endpoint = "/pricing/options"

# Narrow the selection: iso -> state -> utility -> load_profile -> load_zone -> capacity_zone -> voltage
params = {"iso": "PJM", "state": "OH"}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    print("Utilities:", data.get("available_utilities"))
    print("Load zones:", data.get("available_load_zones"))
    print("Capacity zones:", data.get("available_capacity_zones"))
else:
    print(f"Error {response.status_code}: {response.text}")


#### Calculate a fair market price

In [ ]:
endpoint = "/pricing/calculate"

payload = {
    "unit": "MWh",                 # "MWh" (default) or "kWh"; sets the basis for usage, $ adders and the report
    "iso": "PJM", "state": "OH", "utility_name": "AEP Ohio",
    "load_zone": "AEP", "load_profile": "Commercial", "voltage": "Secondary",
    "curve_date": "2025-08-15", "start_date": "2025-09-01", "term_months": 12,
    "plc_kw": 150.0, "nspl_kw": 140.0,
    "monthly_usage": [42, 38, 41, 39, 45, 52, 58, 57, 50, 43, 40, 44],  # exactly 12, in the unit above
    "price_to_compare": 78.50,      # optional, on your unit basis
}

response = requests.post(
    f"{base_url}{endpoint}",
    headers={**headers, "Content-Type": "application/json"},
    json=payload,
)

if response.status_code == 200:
    data = response.json()
    print(f"Fair market price: {data['total_fr_price']} per {data['unit']}")
    print(f"Delta vs price_to_compare: {data.get('delta')} per {data['unit']}")
else:
    print(f"Error {response.status_code}: {response.text}")


### Comparative Savings
Compare an account's supply price against the utility Price to Compare. Requires the **Electricity Utility Price** permission. Limits: 10 requests/minute, 600 accounts/day. Call `/options` first, then `/comparative-analysis`.

#### List priceable combinations

In [ ]:
endpoint = "/api/v1/utility-price/options"

params = {"state": "MD"}   # both filters optional: state, utility_name (exact, case-insensitive)

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    print(f"Dataset {data['operating_date']}: {data['count']} combinations")
    for combo in data["combinations"][:5]:
        cov = combo["coverage"]
        print(f"  {combo['state']} | {combo['utility_name']} | "
              f"{combo['rate_class_load_profile']} | {combo['load_zone']} "
              f"({cov['first_month']} to {cov['last_month']})")
else:
    print(f"Error {response.status_code}: {response.text}")


#### Analyze a portfolio

In [ ]:
endpoint = "/api/v1/utility-price/comparative-analysis"

payload = {
    "reads": [
        {
            "account_id": "ACCT-1001",
            "state": "MD",
            "utility_name": "Baltimore Gas & Electric",
            "rate_class_load_profile": "Residential Service (R)",
            "load_zone": "BGE",
            "service_start": "2026-01-15",
            "service_end": "2026-02-14",
            "convention": "READ_START",   # READ_START | READ_END | DAY_WEIGHTED | LOAD_WEIGHTED
            "usage": 12500,               # kWh for the period
            "supply_price": 0.089,        # $/kWh you pay today; optional
        }
    ]
}

response = requests.post(
    f"{base_url}{endpoint}",
    headers={**headers, "Content-Type": "application/json"},
    json=payload,
    timeout=120,
)

if response.status_code == 200:
    for a in response.json()["accounts"]:
        if not a["supported"]:
            print(f"{a['account_id']}: not priced ({a['reason']})")
            continue
        s = a["summary"]
        print(f"{a['account_id']} (zone {a['header']['load_zone']}): "
              f"utility {s['avg_utility_price']} {a['price_unit']}, "
              f"savings {s['total_savings_dollar']} total")
else:
    print(f"Error {response.status_code}: {response.text}")


### Bulk Download APIs

Bulk download APIs allow you to download data for multiple operating dates in a single Excel file, with each date as a separate sheet.


In [ ]:
## Bulk Utility Price Download

endpoint = "/datasets/download/bulk/utility-price"

params = {
    "start_operating_date": "2025-08-01",        # Required
    "end_operating_date": "2025-08-31",          # Required for bulk download
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_date = params["start_operating_date"]
end_date = params["end_operating_date"]
filename = f"EPD_UtilityPrice_Bulk_{start_date}_to_{end_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f" Bulk file downloaded successfully and saved as '{filename}'")
    print(" Each operating date will be in a separate sheet with Summary and PTC Details")
else:
    print(f" Error {response.status_code}: {response.text}")


In [ ]:
## Bulk Energy Futures Download

endpoint = "/datasets/download/bulk/energy-futures"

params = {
    "start_operating_date": "2025-08-01",        # Required
    "end_operating_date": "2025-08-31",          # Required for bulk download
    "control_area": "ERCOT",                     # Optional: ERCOT, PJM, ISONE, etc.
    "block_types": "7x8,2x16",                   # Optional: filter by block types
    "start_date": "2025-08-01",                  # Optional: additional date filter
    "end_date": "2030-09-01"                     # Optional: additional date filter
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_date = params["start_operating_date"]
end_date = params["end_operating_date"]
control_area = params.get("control_area", "ALL")
filename = f"EPD_EnergyFutures_Bulk_{control_area}_{start_date}_to_{end_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Bulk file downloaded successfully and saved as '{filename}'")
    print("Each operating date will be in a separate sheet")
else:
    print(f"Error {response.status_code}: {response.text}")


In [ ]:
## Bulk REC/RPS Download

endpoint = "/datasets/download/bulk/rec-rps"

params = {
    "start_operating_date": "2025-08-01",        # Required
    "end_operating_date": "2025-08-31",          # Required for bulk download
    "control_area": "ISONE",                     # Optional: ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: additional date filter
    "end_date": "2030-09-01"                     # Optional: additional date filter
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_date = params["start_operating_date"]
end_date = params["end_operating_date"]
control_area = params.get("control_area", "ALL")
filename = f"EPD_REC_RPS_Bulk_{control_area}_{start_date}_to_{end_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Bulk file downloaded successfully and saved as '{filename}'")
    print("Each operating date will be in a separate sheet")
else:
    print(f"Error {response.status_code}: {response.text}")


In [ ]:
## Bulk Ancillary Uplift Download

endpoint = "/datasets/download/bulk/ancillary-uplift"

params = {
    "start_operating_date": "2025-08-01",        # Required
    "end_operating_date": "2025-08-31",          # Required for bulk download
    "control_area": "PJM",                       # Optional: ERCOT, PJM, ISONE, etc.
    "start_date": "2025-08-01",                  # Optional: additional date filter
    "end_date": "2030-09-01"                     # Optional: additional date filter
}

response = requests.get(f"{base_url}{endpoint}", headers=headers, params=params)

start_date = params["start_operating_date"]
end_date = params["end_operating_date"]
control_area = params.get("control_area", "ALL")
filename = f"EPD_AncillaryUplift_Bulk_{control_area}_{start_date}_to_{end_date}.xlsx"

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"Bulk file downloaded successfully and saved as '{filename}'")
    print("Each operating date will be in a separate sheet")
else:
    print(f"Error {response.status_code}: {response.text}")


### Complete API Summary


## Dataset Types Available

#### 1. **Energy Futures** (`/datasets/download/energy-futures`)
- **Excel**: Single or bulk download with multiple sheets
- **CSV**: Single date download
- **JSON**: Single date with pagination support
- **Parameters**: `control_area`, `block_types`, date filters
- **Bulk**: `/datasets/download/bulk/energy-futures`

#### 2. **Utility Price** (`/datasets/download/utility-price`)
- **Excel**: Single or bulk download
- **CSV**: Combined zip file, or separate summary/details endpoints
- **JSON**: Single date with pagination support (`skip`/`limit`)
- **Special Endpoints**:
  - `/datasets/download/utility-price/summary/csv`
  - `/datasets/download/utility-price/details/csv`
- **Bulk**: `/datasets/download/bulk/utility-price`

#### 3. **REC/RPS** (`/datasets/download/rec-rps`)
- **Excel**: Single or bulk download with multiple sheets
- **CSV**: Single date download
- **JSON**: Single date with pagination support
- **Parameters**: `control_area`, date filters
- **Bulk**: `/datasets/download/bulk/rec-rps`

#### 4. **Ancillary Uplift** (`/datasets/download/ancillary-uplift`)
- **Excel**: Single or bulk download with multiple sheets
- **CSV**: Single date download
- **JSON**: Single date with pagination support
- **Parameters**: `control_area`, date filters
- **Bulk**: `/datasets/download/bulk/ancillary-uplift`

#### 5. **Natural Gas Utility Price** (`/datasets/download/naturalgas-utility-price`)
- **Excel**: Single date workbook, or ZIP of per-date xlsx files when `end_operating_date` is set
- **CSV**: Combined ZIP (summary + details), or separate summary/details endpoints
- **JSON**: Single date; `raw=true` returns inline `{summary, details, pagination}` with `offset`/`limit` paging the details list (note: this endpoint uses `offset`, not `skip`)
- **Special Endpoints**:
  - `/datasets/download/naturalgas-utility-price/summary/csv`
  - `/datasets/download/naturalgas-utility-price/details/csv`

###  Common Parameters

#### Required Parameters
- `start_operating_date`: Date in YYYY-MM-DD format

#### Optional Parameters
- `end_operating_date`: For bulk downloads (date range)
- `control_area`: ERCOT, PJM, ISONE, etc.
- `start_date`/`end_date`: Additional date filters
- `raw`: Return JSON data directly (JSON endpoints only)
- `skip`/`limit`: Pagination on Utility Price, REC/RPS, Ancillary Uplift, Energy Futures JSON endpoints
- `offset`/`limit`: Pagination on the Natural Gas Utility Price JSON endpoint
- `block_types`: Energy Futures specific filter

###  Authentication
All endpoints require API key authentication via `X-API-Key` header.

### Complete Coverage
This notebook now includes **ALL** download center APIs:
- 4 Dataset types × 3 formats = 12 standard endpoints
- 4 Bulk download endpoints
- 2 Utility Price specific CSV endpoints
- 5 Natural Gas Utility Price endpoints (Excel, summary CSV, details CSV, combined CSV ZIP, JSON)
- Proper error handling and parameter documentation
- **Total: 23 API endpoints covered**
